In [11]:
from sklearn.datasets import make_moons, load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.calibration import CalibratedClassifierCV 
from sklearn.tree import DecisionTreeClassifier

print('Good imports')

Good imports


In [7]:
X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

log_clf = LogisticRegression(random_state=42)
rnd_clf = RandomForestClassifier(random_state=42)
base_svm = SVC(random_state=42)
calibrated_svm = CalibratedClassifierCV(base_svm, ensemble=False)

voting_clf_soft = VotingClassifier(
    estimators=[('lr', log_clf), ('rf', rnd_clf), ('svc', calibrated_svm)],
    voting='soft'
)

voting_clf_hard = VotingClassifier(
    estimators=[('lr', log_clf), ('rf', rnd_clf), ('svc', calibrated_svm)],
    voting='hard'
)

print("--- Сравнение точности с Мягким голосованием (Soft Voting) ---")
for clf in (log_clf, rnd_clf, calibrated_svm, voting_clf_soft,voting_clf_hard):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(f"{clf.__class__.__name__}: {accuracy_score(y_test, y_pred):.2%}")


--- Сравнение точности с Мягким голосованием (Soft Voting) ---
LogisticRegression: 85.00%
RandomForestClassifier: 88.00%
CalibratedClassifierCV: 90.00%
VotingClassifier: 89.00%
VotingClassifier: 89.00%


In [ ]:
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(random_state=42), n_estimators = 500, 
    max_samples=100, bootstrap=True, n_jobs=-1, oob_score=True, random_state=42
)

bag_clf.fit(X_train, y_train)

print(f"Автоматическая OOB-точность ансамбля: {bag_clf.oob_score_:.2%}")
y_pred = bag_clf.predict(X_test)
print(f"Честная точность на тестовой выборке: {accuracy_score(y_test, y_pred):.2%}")

Автоматическая OOB-точность ансамбля: 91.50%
Честная точность на тестовой выборке: 90.00%


In [10]:
paste_clf = BaggingClassifier(
    DecisionTreeClassifier(random_state=42), n_estimators=500,
    max_samples=100, bootstrap=False, n_jobs=-1, random_state=42
)

paste_clf.fit(X_train, y_train)

y_pred_paste = paste_clf.predict(X_test)
print(f"Точность БЭГГИНГА на тесте: {accuracy_score(y_test, y_pred):.2%}")
print(f"Точность ПЕЙСТИНГА на тестовой выборке: {accuracy_score(y_test, y_pred_paste):.2%}")

Точность БЭГГИНГА на тесте: 90.00%
Точность ПЕЙСТИНГА на тестовой выборке: 91.00%


In [12]:
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42)

subspace_clf = BaggingClassifier(
    DecisionTreeClassifier(random_state=42), n_estimators=500,
    bootstrap=False, max_samples=1.0,
    bootstrap_features=True, max_features=2,
    n_jobs=-1, random_state=42
)

subspace_clf.fit(X_train_i, y_train_i)
y_pred_sub = subspace_clf.predict(X_test_i)

print(f"Точность метода Случайных Подпространств на Ирисах: {accuracy_score(y_test_i, y_pred_sub):.2%}")

Точность метода Случайных Подпространств на Ирисах: 100.00%
